In [2]:
import asyncio
from typing import Annotated, Any, AsyncGenerator, Generator, Optional, Sequence, TypedDict, Union

import mlflow
import nest_asyncio
from databricks.sdk import WorkspaceClient
from databricks_langchain import (
    ChatDatabricks,
    DatabricksMCPServer,
    DatabricksMultiServerMCPClient,
)
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.pyfunc.model import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from langchain_core.messages.tool import ToolMessage
import json



nest_asyncio.apply()
############################################
## Define your LLM endpoint and system prompt
############################################
#LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4-5"
LLM_ENDPOINT_NAME = "databricks-gemma-3-12b"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

# TODO: Update with your system prompt
system_prompt = """
You are a helpful assistant that can run Python code.
"""

# TODO: Choose your MCP server connection type and setup the Workspace Clients for Authentication

# ---------------------------------------------------------------------------
# Managed MCP Server — simplest setup
# ---------------------------------------------------------------------------
# Databricks manages this connection automatically using your workspace settings
# and Personal Access Token (PAT) authentication.

workspace_client = WorkspaceClient()

host = workspace_client.config.host

# ---------------------------------------------------------------------------
# Custom MCP Server — hosted as a Databricks App
# ---------------------------------------------------------------------------
# Use this if you’re running your own MCP server in Databricks.
# These require OAuth with a service principal for machine-to-machine (M2M) auth.
#
# Follow the insturctions here in order to create a SP, grant the SP query permissions on your app and then mint a client id and # secret. https://docs.databricks.com/aws/en/dev-tools/auth/oauth-m2m
#
# Uncomment and fill in the settings below to use a custom MCP server.
#
# import os
# custom_mcp_server_workspace_client = WorkspaceClient(
#     host="<DATABRICKS_WORKSPACE_URL>",
#     client_id=os.getenv("DATABRICKS_CLIENT_ID"),
#     client_secret=os.getenv("DATABRICKS_CLIENT_SECRET"),
#     auth_type="oauth-m2m",  # Enables service principal authentication
# )

# ---------------------------------------------------------------------------
# OBO Setup
# ---------------------------------------------------------------------------
# In order to use OBO, uncomment the code below and pass this workspace client to the appropriate McpServer below
#
# from databricks_ai_bridge import ModelServingUserCredentials
# obo_workspace_client = WorkspaceClient(credentials_strategy=ModelServingUserCredentials())

###############################################################################
## Configure MCP Servers for your agent
##
## This section sets up server connections so your agent can retrieve data or take actions.

## There are three connection types:
## 1. Managed MCP servers — fully managed by Databricks
## 2. External MCP servers — hosted outside Databricks but proxied through a
##    Managed MCP server proxy
## 3. Custom MCP servers — MCP servers hosted as Databricks Apps
##
###############################################################################
databricks_mcp_client = DatabricksMultiServerMCPClient(
    [
        DatabricksMCPServer(
            name="system-ai",
            url=f"{host}/api/2.0/mcp/functions/system/ai",
        ),
        # DatabricksMCPServer(
        #     name="custom_mcp",
        #     url="custom_app_url",
        #     workspace_client=custom_mcp_server_workspace_client
        # ),
        # DatabricksMCPServer(
        #     name="obo_vs_client",
        #     url=f"{host}/api/2.0/mcp/vector-search/system/ai",
        #     workspace_client=obo_workspace_client
        # )
    ]
)


# The state for the agent workflow, including the conversation and any custom data
class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    system_prompt: Optional[str] = None,
):
    model = model.bind_tools(tools)  # Bind tools to the model

    # Function to check if agent should continue or finish based on last message
    def should_continue(state: AgentState):
        messages = state["messages"]
        last_message = messages[-1]
        # If function (tool) calls are present, continue; otherwise, end
        if isinstance(last_message, AIMessage) and last_message.tool_calls:
            return "continue"
        else:
            return "end"

    # Preprocess: optionally prepend a system prompt to the conversation history
    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])

    model_runnable = preprocessor | model  # Chain the preprocessor and the model

    # The function to invoke the model within the workflow
    def call_model(
        state: AgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    workflow = StateGraph(AgentState)  # Create the agent's state machine

    workflow.add_node("agent", RunnableLambda(call_model))  # Agent node (LLM)
    workflow.add_node("tools", ToolNode(tools))  # Tools node

    workflow.set_entry_point("agent")  # Start at agent node
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",  # If the model requests a tool call, move to tools node
            "end": END,  # Otherwise, end the workflow
        },
    )
    workflow.add_edge("tools", "agent")  # After tools are called, return to agent node

    # Compile and return the tool-calling agent workflow
    return workflow.compile()


# ResponsesAgent class to wrap the compiled agent and make it compatible with Databricks Responses API
class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, agent):
        self.agent = agent

    # Make a prediction (single-step) for the agent
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done" or event.type == "error"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    async def _predict_stream_async(
        self,
        request: ResponsesAgentRequest,
    ) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])
        # Stream events from the agent graph
        async for event in self.agent.astream(
            {"messages": cc_msgs}, stream_mode=["updates", "messages"]
        ):
            if event[0] == "updates":
                # Stream updated messages from the workflow nodes
                for node_data in event[1].values():
                    if len(node_data.get("messages", [])) > 0:
                        all_messages = []
                        for msg in node_data["messages"]:
                            if isinstance(msg, ToolMessage) and not isinstance(msg.content, str):
                                msg.content = json.dumps(msg.content)
                            all_messages.append(msg)
                        for item in output_to_responses_items_stream(all_messages):
                            yield item
            elif event[0] == "messages":
                # Stream generated text message chunks
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=content, item_id=chunk.id),
                        )
                except:
                    pass

    # Stream predictions for the agent, yielding output as it's generated
    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        agen = self._predict_stream_async(request)

        try:
            loop = asyncio.get_event_loop()
        except RuntimeError:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)

        ait = agen.__aiter__()

        while True:
            try:
                item = loop.run_until_complete(ait.__anext__())
            except StopAsyncIteration:
                break
            else:
                yield item


# Initialize the entire agent, including MCP tools and workflow
def initialize_agent():
    """Initialize the agent with MCP tools"""
    # Create MCP tools from the configured servers
    mcp_tools = asyncio.run(databricks_mcp_client.get_tools())

    # Create the agent graph with an LLM, tool set, and system prompt (if given)
    agent = create_tool_calling_agent(llm, mcp_tools, system_prompt)
    return LangGraphResponsesAgent(agent)


mlflow.langchain.autolog()
AGENT = initialize_agent()
mlflow.models.set_model(AGENT)

### Test the agent

In [ ]:
# ==============================================================================
# TODO: ONLY UNCOMMENT AND EDIT THIS SECTION IF YOU ARE USING OAUTH/SERVICE PRINCIPAL FOR CUSTOM MCP SERVERS.
#       For managed MCP (the default), LEAVE THIS SECTION COMMENTED OUT.
# ==============================================================================

# import os

# # Set your Databricks client ID and client secret for service principal authentication.
# DATABRICKS_CLIENT_ID = "<YOUR_CLIENT_ID>"
# client_secret_scope_name = "<YOUR_SECRET_SCOPE>"
# client_secret_key_name = "<YOUR_SECRET_KEY_NAME>"

# # Load your service principal credentials into environment variables
# os.environ["DATABRICKS_CLIENT_ID"] = DATABRICKS_CLIENT_ID
# os.environ["DATABRICKS_CLIENT_SECRET"] = dbutils.secrets.get(scope=client_secret_scope_name, key=client_secret_key_name)

In [ ]:
from agent import AGENT

AGENT.predict({"input": [{"role": "user", "content": "What is 7*6 in Python?"}]})

2026/05/31 16:34:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/31 16:34:46 INFO mlflow.store.db.utils: Updating database tables


ResponsesAgentResponse(tool_choice=None, truncation=None, id=None, created_at=None, error=None, incomplete_details=None, instructions=None, metadata=None, model=None, object='response', output=[OutputItem(type='function_call', id='call_6ac42536-9353-4886-be42-b5e52652b02d', call_id='call_6ac42536-9353-4886-be42-b5e52652b02d', name='system__ai__python_exec', arguments='{"code": "print(7*6)"}'), OutputItem(type='function_call_output', call_id='call_6ac42536-9353-4886-be42-b5e52652b02d', output='[{"type": "text", "text": "{\\"is_truncated\\":false,\\"columns\\":[\\"output\\"],\\"rows\\":[[\\"42\\\\n\\"]]}", "id": "lc_31b19198-4af2-4ef3-b829-556a75488277"}]'), OutputItem(type='message', id='lc_run--019e7db5-d759-7140-beff-37f3488c2b89', content=[{'text': '7 * 6 is 42.', 'type': 'output_text', 'annotations': []}], role='assistant')], parallel_tool_calls=None, temperature=None, tools=None, top_p=None, max_output_tokens=None, previous_response_id=None, reasoning=None, status=None, text=None, 

In [5]:
for chunk in AGENT.predict_stream(
    {"input": [{"role": "user", "content": "What is 7*6 in Python?"}]}
):
    print(chunk, "-----------\n")

type='response.output_item.done' custom_outputs=None item={'type': 'function_call', 'id': 'call_60d298ab-adff-4b64-8f1b-efc96a587c1a', 'call_id': 'call_60d298ab-adff-4b64-8f1b-efc96a587c1a', 'name': 'system__ai__python_exec', 'arguments': '{"code": "print(7*6)"}'} -----------

type='response.output_item.done' custom_outputs=None item={'type': 'function_call_output', 'call_id': 'call_60d298ab-adff-4b64-8f1b-efc96a587c1a', 'output': '[{"type": "text", "text": "{\\"is_truncated\\":false,\\"columns\\":[\\"output\\"],\\"rows\\":[[\\"42\\\\n\\"]]}", "id": "lc_87d1ed0f-e887-47c5-8cdc-553357a92e52"}]'} -----------

type='response.output_text.delta' custom_outputs=None item_id='lc_run--019e7db6-b718-7c40-bc38-4ac8a66a2a1a' delta='7*6 is 42.' -----------

type='response.output_item.done' custom_outputs=None item={'id': 'lc_run--019e7db6-b718-7c40-bc38-4ac8a66a2a1a', 'content': [{'text': '7*6 is 42.', 'type': 'output_text', 'annotations': []}], 'role': 'assistant', 'type': 'message'} -----------


### Log the agent as an MLflow model

In [8]:

import mlflow
#from agent import LLM_ENDPOINT_NAME
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction
#from pkg_resources import get_distribution
from importlib.metadata import version
#my_version = version("my-package")

resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME), 
    DatabricksFunction(function_name="system.ai.python_exec")
]

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="src/main/py/agent.py",
        resources=resources,
        pip_requirements=[
            f"langgraph=={version('langgraph')}",
            f"mcp=={version('mcp')}",
            f"databricks-mcp=={version('databricks-mcp')}",
            f"databricks-langchain=={version('databricks-langchain')}",
        ]
    )

2026/05/31 16:43:19 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/brijeshdhaker/IdeaProjects/bd-databricks-module


2026/05/31 16:43:26 INFO mlflow.pyfunc: Predicting on input example to validate output
2026/05/31 16:43:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/31 16:43:30 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/brijeshdhaker/IdeaProjects/bd-databricks-module
2026/05/31 16:43:30 INFO mlflow.utils.uv_utils: Copied uv.lock to model artifacts
2026/05/31 16:43:30 INFO mlflow.utils.uv_utils: Copied pyproject.toml to model artifacts
2026/05/31 16:43:37 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN, OPENAI_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


### Evaluate the agent with Agent Evaluation
You can edit the requests or expected responses in your evaluation dataset and run evaluation as you iterate your agent, leveraging mlflow to track the computed quality metrics.

Evaluate your agent with one of our predefined LLM scorers, or try adding custom metrics.

In [9]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, Safety, RetrievalRelevance, RetrievalGroundedness

eval_dataset = [
    {
        "inputs": {
            "input": [
                {
                    "role": "user",
                    "content": "Calculate the 15th Fibonacci number"
                }
            ]
        },
        "expected_response": "The 15th Fibonacci number is 610."
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()], # add more scorers here if they're applicable
)

# Review the evaluation results in the MLfLow UI (see console output)

2026/05/31 16:44:02 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/05/31 16:44:02 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?]

2026/05/31 16:44:38 INFO mlflow.genai.evaluation.rate_limiter: Rate limit hit — reducing rate from 20.0 to 10.0 rps
2026/05/31 16:44:38 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 1/4), retrying in 1s
2026/05/31 16:44:38 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 1/4), retrying in 1s
2026/05/31 16:44:40 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 2/4), retrying in 2s
2026/05/31 16:44:40 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 2/4), retrying in 2s
2026/05/31 16:44:42 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 3/4), retrying in 4s
2026/05/31 16:44:42 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 3/4), retrying in 4s
2026/05/31 16:44:47 INFO mlflow.genai.evaluation.rate_limiter: Rate limit hit — reducing rate from 10.0 to 5.0 rps


AttributeError: 'NoneType' object has no attribute 'info'

In [ ]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "What is 7*6 in Python?"}]},
    env_manager="uv",
)

### Register the model to Unity Catalog
Before you deploy the agent, you must register the agent to Unity Catalog.

TODO Update the catalog, schema, and model_name below to register the MLflow model to Unity Catalog.

In [ ]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "workspace"
schema = "default"
model_name = "dbx-test-model"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, 
    name=UC_MODEL_NAME
)

### Deploy the agent

In [ ]:
from databricks import agents

agents.deploy(
    UC_MODEL_NAME, 
    uc_registered_model_info.version,
    # ==============================================================================
    # TODO: ONLY UNCOMMENT AND CONFIGURE THE ENVIRONMENT_VARS SECTION BELOW
    #       IF YOU ARE USING OAUTH/SERVICE PRINCIPAL FOR CUSTOM MCP SERVERS.
    #       For managed MCP (the default), LEAVE THIS SECTION COMMENTED OUT.
    # ==============================================================================
    # environment_vars={
    #     "DATABRICKS_CLIENT_ID": DATABRICKS_CLIENT_ID,
    #     "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{client_secret_scope_name}/{client_secret_key_name}}}}}"
    # },
    tags = {"endpointSource": "docs"},
    deploy_feedback_model=False
)